In [7]:
import ir_datasets, pandas as pd
import numpy as np

ds = ir_datasets.load("beir/nfcorpus/test")
docs = pd.DataFrame(ds.docs_iter())
queries = pd.DataFrame(ds.queries_iter())
qrels = pd.DataFrame(ds.qrels_iter())

[INFO] Opening /Users/bhagya/.ir_datasets/beir/nfcorpus/docs.pklz4/bin with direct file access


In [8]:
rel = {qid: dict(zip(g.doc_id, g.relevance)) for qid, g in qrels[qrels.relevance > 0].groupby("query_id")}

In [9]:
def rr(ids, relevant):
    for i, d in enumerate(ids, 1):
        if d in relevant:
            return 1 / i
    return 0


In [10]:
def evaluate(search_fn, k=20):
    rows = []
    for q in queries.itertuples():
        if q.query_id in rel:
            ids = [d for d, _ in search_fn(q.text, k)]
            rows.append((q.query_id, rr(ids, rel[q.query_id])))
    return pd.DataFrame(rows, columns=["query_id", "rr"])


In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
corpus = (docs.title + " " + docs.text).tolist()
emb = model.encode(corpus, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

def dense_search(q, k=20):
    s = emb @ model.encode(q, normalize_embeddings=True)
    idx = np.argsort(-s)[:k]
    return [(docs.doc_id[i], s[i]) for i in idx]



Batches: 100%|██████████| 57/57 [01:17<00:00,  1.37s/it]


In [12]:
dense_res = evaluate(dense_search)
dense_res.rr.mean()

np.float64(0.532641881634252)